# EU4 决策模型训练（Colab GPU）

说明：在 Colab GPU 上跑 EU4 决策模型（`TerritoryGNN`）训练。整个流程：安装 Rust 工具链 → 上传训练包 → 安装 PyTorch Geometric → 编译 Rust 环境服务器（`territory_server`）→ 用 `_territory_raft.py` 做 advantage-clone（GRPO 风格）训练 → 下载权重 `gnn.pt`。

准备工作：在本机仓库根目录已生成 `target/foch-eu4-decide-bundle.tar.gz`（只含编译+训练所需的最小工作区，几百 KB，不需要 clone 整个仓库——仓库里有体积很大的游戏数据）。运行本 notebook 前请选择 GPU 运行时（代码执行程序 → 更改运行时类型 → T4 GPU）。

## 1. 安装 Rust 工具链

工作区使用 **edition 2024**（`rust-version = 1.95`），而 Ubuntu apt 源里的 rustc 太旧（edition 2024 至少需要 rustc 1.85，apt 通常只有 1.7x），所以这里用 rustup 安装官方工具链，而不是 `apt-get install cargo rustc`。

In [ ]:
# 备选方案：如果将来 apt 的 cargo 足够新（>= 1.85），可以用
# !apt-get -qq install -y cargo rustc
# 这里用 rustup 安装满足 edition 2024 的官方工具链：
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!rustc --version && cargo --version

## 2. 上传训练包并解压

不 clone 仓库（游戏数据太大），改为上传 `foch-eu4-decide-bundle.tar.gz`。若已挂载 Google Drive 且云盘根目录有同名文件，则直接复制，不用手动上传；否则弹出上传框选择本地文件。解压到 `/content/foch`。

In [ ]:
import os
from google.colab import files

BUNDLE = '/content/foch-eu4-decide-bundle.tar.gz'
DRIVE_COPY = '/content/drive/MyDrive/foch-eu4-decide-bundle.tar.gz'

if os.path.exists(DRIVE_COPY):
    # 云盘已有 bundle，直接复制。若尚未挂载 Drive，先运行：
    # from google.colab import drive; drive.mount('/content/drive')
    !cp {DRIVE_COPY} {BUNDLE}
    print('已从 Google Drive 复制 bundle')
elif os.path.exists(BUNDLE):
    print('bundle 已存在，跳过上传')
else:
    uploaded = files.upload()  # 选择本机的 foch-eu4-decide-bundle.tar.gz
    assert 'foch-eu4-decide-bundle.tar.gz' in uploaded, '请上传 foch-eu4-decide-bundle.tar.gz'

!tar -xzf {BUNDLE} -C /content
!ls /content/foch

## 3. 安装 PyTorch Geometric

Colab 已预装 torch。`torch_geometric` 直接 `pip install` 即可——训练用到的 `GATv2Conv` 是纯 PyTorch 实现，**不需要**再装 torch-scatter / torch-sparse / torch-cluster 等扩展 wheel。

In [ ]:
!pip install -q torch_geometric
import torch, torch_geometric
print('torch', torch.__version__, '| pyg', torch_geometric.__version__, '| cuda:', torch.cuda.is_available())

## 4. 编译环境服务器

训练包里带了一个最小化的 cargo workspace，只含 `foch-eu4-decide` 一个成员。编译 `territory_server`（stdin/stdout JSON 协议的环境服务器）。

In [ ]:
!cd /content/foch && cargo build --release -p foch-eu4-decide --example territory_server
!ls /content/foch/target/release/examples/

## 5. 训练（GPU）

`--device cuda` 只加速训练部分（批量前向/反向）；数据采集（跑环境）仍在 CPU 上解析状态。

注意工作目录：训练脚本没有 `--provinces` 参数，环境服务器按默认路径查找省份表——先 `%cd /content/foch/src/tools/eu4-decide`，服务器的回退路径 `../../../target/eu4_provinces.txt` 就会解析到 bundle 里的 `/content/foch/target/eu4_provinces.txt`（真实地图）。如果日志出现 `no province pool, falling back to synthetic maps`，说明工作目录不对。

参数可按需调整：`--episodes/--rollouts` 决定每轮数据量，`--rounds/--epochs` 决定训练总时长；显存或时间不够就先调小 `--episodes`。

In [ ]:
%%time
%cd /content/foch/src/tools/eu4-decide
!python -m eu4_decide._territory_raft \
    --env /content/foch/target/release/examples/territory_server \
    --device cuda --advantage-clone \
    --episodes 32 --rollouts 8 --rounds 6 --epochs 150 \
    --weights /content/foch/gnn.pt

## 6. 下载权重

训练脚本始终保存 CPU 张量（best round 的权重），下载后可在任何机器上 `TerritoryGNN().load_state_dict(torch.load('gnn.pt'))` 直接使用。

In [ ]:
from google.colab import files
files.download('/content/foch/gnn.pt')